In [29]:
import os
import numpy as np
import pandas as pd
from typing import Dict, List
import time
from dotenv import load_dotenv
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from financerag.task import *
from financerag.retrieval import DenseRetrieval
from langchain_text_splitters.base import TextSplitter
from financerag.common import get_query_and_retrieved_corpus_text, process_retrieval_df, get_final_result 
from sentence_transformers import CrossEncoder
from transformers import pipeline
pd.set_option('display.max_colwidth', 400)
import warnings
warnings.filterwarnings('ignore')


In [3]:
load_dotenv(".env")
GG_API_KEY = os.environ.get('GOOGLE_API_KEY')
PINECONE_API_KEY = os.environ.get('PINECONE_API_KEY')

In [ ]:
def load_query(dataset_name : str, new_path_to_query = None) -> Dict[str, str]:
    query_df = pd.read_json(f"finance_dataset/{dataset_name.lower()}_queries.jsonl/queries.jsonl", lines = True)
    # Convert into Dict[str, str]
    query_dict = {row['_id'] : row['text'] for i, row in query_df.iterrows()}
    return query_dict

    
def load_corpus(dataset_name : str, new_path_to_corpus = None) -> Dict[str, str]:
    corpus_df = pd.read_json(f"finance_dataset/{dataset_name.lower()}_corpus.jsonl/corpus.jsonl", lines = True)
    # Convert into Dict[str, str]
    corpus_dict = {row['_id'] : row['text'] for i, row in corpus_df.iterrows()}
    return corpus_dict

In [37]:
# Set up vector database
import faiss
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_community.vectorstores import FAISS
embeddings = GoogleGenerativeAIEmbeddings(model = "models/text-embedding-004")
index = faiss.IndexFlatL2(len(embeddings.embed_query("hello world")))

vector_store = FAISS(
    embedding_function=embeddings,
    index=index,
    docstore=InMemoryDocstore(),
    index_to_docstore_id={},
)

In [6]:
import torch # type: ignore
torch.__version__

'2.3.0.dev20240311'

In [7]:
if torch.backends.mps.is_available():
    mps_device = torch.device("mps")

device = 'mps' if torch.backends.mps.is_available() else 'cpu'
device

'mps'

In [ ]:
# Get dataset name first
import os
dataset_names = []
for f in os.listdir('finance_dataset'):
    if f.endswith("tsv"):
       dataset_names.append(f.split('_')[0])
dataset_names  

['MultiHeirtt',
 'FinQA',
 'FinanceBench',
 'ConvFinQA',
 'FinQABench',
 'TATQA',
 'FinDER']

In [15]:
CHUNK_SIZE = 2000
CHUNK_OVERLAP = 300
text_splitter = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)

In [ ]:
for dataset_name in dataset_names:
    task_variable = f"{dataset_name.lower()}_task"
    script_string = f"""
    # {dataset_name} Task
    print(f"{dataset_name} task")
    {task_variable} = {dataset_name}Task()
    {task_variable}.load()
    {task_variable}_max_len = np.max([len(text) for text in {task_variable}.corpus.values()])
    {task_variable}_top_k = ({task_variable}_max_len // (CHUNK_SIZE - CHUNK_OVERLAP) + 1) * 10 + 10
    {task_variable}_retriever = DenseRetrieval(vector_store = vector_store, dataset_name = {task_variable}.metadata.dataset_name)
    {task_variable}_retriever.load_corpus_with_splitting(text_splitter = text_splitter, corpus = {task_variable}.corpus, saved_index = False)
    retrieved_result = {task_variable}.retrieve(retriever = {task_variable}_retriever, top_k = {task_variable}_top_k)
    {task_variable}.save_retrieved_results(retrieved_result = retrieved_result)
    time.sleep(60)
    """
    print(script_string)


    # MultiHeirtt Task
    print(f"MultiHeirtt task")
    multiheirtt_task = MultiHeirttTask()
    multiheirtt_task.load()
    multiheirtt_task_max_len = np.max([len(text) for text in multiheirtt_task.corpus.values()])
    multiheirtt_task_top_k = (multiheirtt_task_max_len // (CHUNK_SIZE - CHUNK_OVERLAP) + 1) * 10 + 10
    multiheirtt_task_retriever = DenseRetrieval(vector_store = vector_store, dataset_name = multiheirtt_task.metadata.dataset_name)
    multiheirtt_task_retriever.load_corpus_with_splitting(text_splitter = text_splitter, corpus = multiheirtt_task.corpus, saved_index = False)
    retrieved_result = multiheirtt_task.retrieve(retriever = multiheirtt_task_retriever, top_k = multiheirtt_task_top_k)
    multiheirtt_task.save_retrieved_results(retrieved_result = retrieved_result)
    time.sleep(60)
    

    # FinQA Task
    print(f"FinQA task")
    finqa_task = FinQATask()
    finqa_task.load()
    finqa_task_max_len = np.max([len(text) for text in finqa_task.corpus.values()]

In [202]:
device = 'mps' if torch.backends.mps.is_available() else 'cpu'
device

'mps'

In [ ]:
# FinQA Task
print(f"FinQA task")
finqa_task = FinQATask()
finqa_task.load()
finqa_task_max_len = np.max([len(text) for text in finqa_task.corpus.values()])
finqa_task_top_k = (finqa_task_max_len // (CHUNK_SIZE - CHUNK_OVERLAP) + 1) * 10 + 10
finqa_task_retriever = DenseRetrieval(vector_store = vector_store, dataset_name = finqa_task.metadata.dataset_name)
finqa_task_retriever.load_corpus_with_splitting(text_splitter = text_splitter, corpus = finqa_task.corpus, saved_index = False)
retrieved_result = finqa_task.retrieve(retriever = finqa_task_retriever, top_k = finqa_task_top_k)
finqa_task.save_retrieved_results(retrieved_result = retrieved_result)
time.sleep(60)


# FinanceBench Task
print(f"FinanceBench task")
financebench_task = FinanceBenchTask()
financebench_task.load()
financebench_task_max_len = np.max([len(text) for text in financebench_task.corpus.values()])
financebench_task_top_k = (financebench_task_max_len // (CHUNK_SIZE - CHUNK_OVERLAP) + 1) * 10 + 10
financebench_task_retriever = DenseRetrieval(vector_store = vector_store, dataset_name = financebench_task.metadata.dataset_name)
financebench_task_retriever.load_corpus_with_splitting(text_splitter = text_splitter, corpus = financebench_task.corpus, saved_index = False)
retrieved_result = financebench_task.retrieve(retriever = financebench_task_retriever, top_k = financebench_task_top_k)
financebench_task.save_retrieved_results(retrieved_result = retrieved_result)
time.sleep(60)

In [ ]:

# ConvFinQA Task
print(f"ConvFinQA task")
convfinqa_task = ConvFinQATask()
convfinqa_task.load()
convfinqa_task_max_len = np.max([len(text) for text in convfinqa_task.corpus.values()])
convfinqa_task_top_k = (convfinqa_task_max_len // (CHUNK_SIZE - CHUNK_OVERLAP) + 1) * 10 + 10
convfinqa_task_retriever = DenseRetrieval(vector_store = vector_store, dataset_name = convfinqa_task.metadata.dataset_name)
convfinqa_task_retriever.load_corpus_with_splitting(text_splitter = text_splitter, corpus = convfinqa_task.corpus, saved_index = False)
retrieved_result = convfinqa_task.retrieve(retriever = convfinqa_task_retriever, top_k = convfinqa_task_top_k)
convfinqa_task.save_retrieved_results(retrieved_result = retrieved_result)
time.sleep(60)


# FinQABench Task
print(f"FinQABench task")
finqabench_task = FinQABenchTask()
finqabench_task.load()
finqabench_task_max_len = np.max([len(text) for text in finqabench_task.corpus.values()])
finqabench_task_top_k = (finqabench_task_max_len // (CHUNK_SIZE - CHUNK_OVERLAP) + 1) * 10 + 10
finqabench_task_retriever = DenseRetrieval(vector_store = vector_store, dataset_name = finqabench_task.metadata.dataset_name)
finqabench_task_retriever.load_corpus_with_splitting(text_splitter = text_splitter, corpus = finqabench_task.corpus, saved_index = False)
retrieved_result = finqabench_task.retrieve(retriever = finqabench_task_retriever, top_k = finqabench_task_top_k)
finqabench_task.save_retrieved_results(retrieved_result = retrieved_result)
time.sleep(60)


# TATQA Task
print(f"TATQA task")
tatqa_task = TATQATask()
tatqa_task.load()
tatqa_task_max_len = np.max([len(text) for text in tatqa_task.corpus.values()])
tatqa_task_top_k = (tatqa_task_max_len // (CHUNK_SIZE - CHUNK_OVERLAP) + 1) * 10 + 10
tatqa_task_retriever = DenseRetrieval(vector_store = vector_store, dataset_name = tatqa_task.metadata.dataset_name)
tatqa_task_retriever.load_corpus_with_splitting(text_splitter = text_splitter, corpus = tatqa_task.corpus, saved_index = False)
retrieved_result = tatqa_task.retrieve(retriever = tatqa_task_retriever, top_k = tatqa_task_top_k)
tatqa_task.save_retrieved_results(retrieved_result = retrieved_result)
time.sleep(60)


# FinDER Task
print(f"FinDER task")
finder_task = FinDERTask()
finder_task.load()
finder_task_max_len = np.max([len(text) for text in finder_task.corpus.values()])
finder_task_top_k = (finder_task_max_len // (CHUNK_SIZE - CHUNK_OVERLAP) + 1) * 10 + 10
finder_task_retriever = DenseRetrieval(vector_store = vector_store, dataset_name = finder_task.metadata.dataset_name)
finder_task_retriever.load_corpus_with_splitting(text_splitter = text_splitter, corpus = finder_task.corpus, saved_index = False)
retrieved_result = finder_task.retrieve(retriever = finder_task_retriever, top_k = finder_task_top_k)
finder_task.save_retrieved_results(retrieved_result = retrieved_result)


TATQA task


Retrieving result:: 100%|██████████| 1663/1663 [10:58<00:00,  2.52it/s]


Saved result successfully to ./financerag_result/tatqa_result.csv!
FinDER task


Retrieving result:: 100%|██████████| 216/216 [01:40<00:00,  2.15it/s]

Saved result successfully to ./financerag_result/finder_result.csv!


In [ ]:
# MultiHeirtt Task
multiheirtt_task = MultiHeirttTask()
multiheirtt_task.load()
multiheirtt_task_retriever = DenseRetrieval(vector_store = vector_store, dataset_name = multiheirtt_task.metadata.dataset_name)
multiheirtt_task_retriever.load_corpus_with_splitting(text_splitter = text_splitter, corpus = multiheirtt_task.corpus, saved_index = True)
multiheirtt_task.retrieve(retriever = multiheirtt_task_retriever)
multiheirtt_task.save_retrieved_results()



Loading document:: 100%|██████████| 10475/10475 [00:00<00:00, 42579.02it/s]


Successfully saved index of vector store to path : faiss_index/multiheirtt_index
Saved result successfully to ./financerag_result/multiheirtt_result.csv!


In [ ]:
# FinQA Task
finqa_task = FinQATask()
finqa_task.load()
finqa_task_retriever = DenseRetrieval(vector_store = vector_store, dataset_name = finqa_task.metadata.dataset_name)
finqa_task_retriever.load_corpus_for_searching_without_splitting(finqa_task.corpus, saved_index = True)
finqa_task.retrieve(retriever = finqa_task_retriever)
finqa_task.save_retrieved_results()




Loading document: 100%|██████████| 2789/2789 [00:00<00:00, 151775.10it/s]


Successfully saved index of vector store to path : faiss_index/finqa_index


Retrieving result:: 100%|██████████| 1147/1147 [07:21<00:00,  2.60it/s]

Saved result successfully to ./financerag_result/finqa_result.csv!


In [ ]:

# FinanceBench Task
financebench_task = FinanceBenchTask()
financebench_task.load()
financebench_task_retriever = DenseRetrieval(vector_store = vector_store, dataset_name = financebench_task.metadata.dataset_name)
financebench_task_retriever.load_corpus_for_searching(financebench_task.corpus, saved_index = True)
financebench_task.retrieve(retriever = financebench_task_retriever)
financebench_task.save_retrieved_results()




Loading document: 100%|██████████| 180/180 [00:00<00:00, 131942.45it/s]


Successfully saved index of vector store to path : faiss_index/financebench_index


Retrieving result:: 100%|██████████| 150/150 [01:02<00:00,  2.42it/s]

Saved result successfully to ./financerag_result/financebench_result.csv!


In [ ]:
# ConvFinQA Task
convfinqa_task = ConvFinQATask()
convfinqa_task.load()
convfinqa_task_retriever = DenseRetrieval(vector_store = vector_store, dataset_name = convfinqa_task.metadata.dataset_name)
convfinqa_task_retriever.load_corpus_for_searching_without_splitting(convfinqa_task.corpus, saved_index = True)
convfinqa_task.retrieve(retriever = convfinqa_task_retriever)
convfinqa_task.save_retrieved_results()


Loading document: 100%|██████████| 2066/2066 [00:00<00:00, 129481.68it/s]


Successfully saved index of vector store to path : faiss_index/convfinqa_index


Retrieving result:: 100%|██████████| 421/421 [02:37<00:00,  2.67it/s]

Saved result successfully to ./financerag_result/convfinqa_result.csv!


In [ ]:

# FinQABench Task
finqabench_task = FinQABenchTask()
finqabench_task.load()
finqabench_task_retriever = DenseRetrieval(vector_store = vector_store, dataset_name = finqabench_task.metadata.dataset_name)
finqabench_task_retriever.load_corpus_for_searching(finqabench_task.corpus, saved_index = True)
finqabench_task.retrieve(retriever = finqabench_task_retriever)
finqabench_task.save_retrieved_results()


Loading document: 100%|██████████| 92/92 [00:00<00:00, 89592.75it/s]


Successfully saved index of vector store to path : faiss_index/finqabench_index


Retrieving result:: 100%|██████████| 100/100 [00:36<00:00,  2.75it/s]

Saved result successfully to ./financerag_result/finqabench_result.csv!


In [ ]:

# TATQA Task
tatqa_task = TATQATask()
tatqa_task.load()
tatqa_task_retriever = DenseRetrieval(vector_store = vector_store, dataset_name = tatqa_task.metadata.dataset_name)
tatqa_task_retriever.load_corpus_for_searching_without_splitting(tatqa_task.corpus, saved_index = True)
tatqa_task.retrieve(retriever = tatqa_task_retriever)
tatqa_task.save_retrieved_results() 


Loading document: 100%|██████████| 2756/2756 [00:00<00:00, 182551.12it/s]


Successfully saved index of vector store to path : faiss_index/tatqa_index


Retrieving result:: 100%|██████████| 1663/1663 [15:14<00:00,  1.82it/s] 

Saved result successfully to ./financerag_result/tatqa_result.csv!


In [ ]:

# FinDER Task
finder_task = FinDERTask()
finder_task.load()
finder_task_retriever = DenseRetrieval(vector_store = vector_store, dataset_name = finder_task.metadata.dataset_name)
finder_task_retriever.load_corpus_for_searching_without_splitting(finder_task.corpus, saved_index = True)
finder_task.retrieve(retriever = finder_task_retriever)
finder_task.save_retrieved_results()

Loading document: 100%|██████████| 13862/13862 [00:00<00:00, 20056.98it/s]


Successfully saved index of vector store to path : faiss_index/finder_index


Retrieving result:: 100%|██████████| 216/216 [01:57<00:00,  1.84it/s]

Saved result successfully to ./financerag_result/finder_result.csv!


In [32]:
method_name = 'corpus_summarization_bm25_reranking'
final_result = get_final_result(dataset_names, method_name = method_name)

In [34]:
final_result

,query_id,corpus_id
0,q82d4c6ec,d8f70d54e
1,q82d4c6ec,d8d3fbbaa
2,q82d4c6ec,d81a04fe4
3,q82d4c6ec,d8177aaf8
4,q82d4c6ec,d887d2b66
...,...,...
2155,q00218,BRK.A20231590
2156,q00218,BRK.A20232058
2157,q00218,BRK.A20230780
2158,q00218,BRK.A20230642


In [36]:
pd.read_csv('submission_bm25_with_reranker.csv')

,query_id,corpus_id
0,q82d4c6ec,d81a04f9e
1,q82d4c6ec,d81a04fe4
2,q82d4c6ec,d8823b39c
3,q82d4c6ec,d8e435ffc
4,q82d4c6ec,d8c9cf7da
...,...,...
46705,q00218,V20230334
46706,q00218,MSFT20231781
46707,q00218,LIN20231761
46708,q00218,GOOGL20231537


In [35]:
final_result.to_csv(f'submission_{method_name}.csv', index = False)

In [22]:
final_result.drop_duplicates(subset = ['query_id'])

,query_id,corpus_id
0,q82d4c6ec,d8e404704
10,q855a35a0,d89a6ea36
20,q85384530,d8ce7fc30
30,q842c8af2,d88465f0a
40,q85451756,d8646ec7e
...,...,...
2094,q00214,BRK.A20230009
2104,q00215,BRK.A20230404
2114,q00216,BRK.A20232401
2124,q00217,BRK.A20230062


In [11]:
import torch.nn as nn

In [ ]:
for dataset_name in dataset_names:
    task_variable = f"{dataset_name.lower()}_task"
    script_string = f"""
    # {dataset_name} Task
    print(f"{dataset_name} task")
    {task_variable} = {dataset_name}Task()
    {task_variable}.load('finance_corpus_summarized')
    {task_variable}_reranker = CrossEncoderReranker(queries = {task_variable}.queries, corpus = {task_variable}.corpus, reranker = model)
    {task_variable}_bm25_retriever = BM25_Retriever()
    retrieved_result = {task_variable}.retrieve(retriever = {task_variable}_bm25_retriever, corpus = {task_variable}.corpus, top_k = 50)
    final_result = {task_variable}_reranker.rerank(retrieved_result, top_k = 10)
    {task_variable}.save_retrieved_results(retrieved_result = final_result)
    """
    print(script_string)


    # MultiHeirtt Task
    print(f"MultiHeirtt task")
    multiheirtt_task = MultiHeirttTask()
    multiheirtt_task.load('finance_corpus_summarized')
    multiheirtt_task_reranker = CrossEncoderReranker(queries = multiheirtt_task.queries, corpus = multiheirtt_task.corpus, reranker = model)
    multiheirtt_task_bm25_retriever = BM25_Retriever()
    retrieved_result = multiheirtt_task.retrieve(retriever = multiheirtt_task_bm25_retriever, corpus = multiheirtt_task.corpus, top_k = 50)
    final_result = multiheirtt_task_reranker.rerank(retrieved_result, top_k = 10)
    multiheirtt_task.save_retrieved_results(retrieved_result = final_result)
    

    # FinQA Task
    print(f"FinQA task")
    finqa_task = FinQATask()
    finqa_task.load('finance_corpus_summarized')
    finqa_task_reranker = CrossEncoderReranker(queries = finqa_task.queries, corpus = finqa_task.corpus, reranker = model)
    finqa_task_bm25_retriever = BM25_Retriever()
    retrieved_result = finqa_task.retrieve(retriever = f

In [22]:
from financerag.retrieval import BM25, BM25_Retriever
from financerag.rerank import CrossEncoderReranker


In [ ]:
for query_id, doc_dict in final_result.items():
    for corpus_id in doc_dict.keys():
        doc_dict[corpus_id] = 1.0
final_result

In [11]:
from sentence_transformers import CrossEncoder

In [12]:
model = CrossEncoder("cross-encoder/ms-marco-MiniLM-L6-v2")

In [24]:
# MultiHeirtt Task
print(f"MultiHeirtt task")
multiheirtt_task = MultiHeirttTask()
multiheirtt_task.load('finance_corpus_summarized')
multiheirtt_task_reranker = CrossEncoderReranker(queries = multiheirtt_task.queries, corpus = multiheirtt_task.corpus, reranker = model)
multiheirtt_task_bm25_retriever = BM25_Retriever()
retrieved_result = multiheirtt_task.retrieve(retriever = multiheirtt_task_bm25_retriever, corpus = multiheirtt_task.corpus, top_k = 50)
final_result = multiheirtt_task_reranker.rerank(retrieved_result, top_k = 10)
multiheirtt_task.save_retrieved_results(retrieved_result = final_result)


# FinQA Task
print(f"FinQA task")
finqa_task = FinQATask()
finqa_task.load('finance_corpus_summarized')
finqa_task_reranker = CrossEncoderReranker(queries = finqa_task.queries, corpus = finqa_task.corpus, reranker = model)
finqa_task_bm25_retriever = BM25_Retriever()
retrieved_result = finqa_task.retrieve(retriever = finqa_task_bm25_retriever, corpus = finqa_task.corpus, top_k = 50)
final_result = finqa_task_reranker.rerank(retrieved_result, top_k = 10)
finqa_task.save_retrieved_results(retrieved_result = final_result)


# FinanceBench Task
print(f"FinanceBench task")
financebench_task = FinanceBenchTask()
financebench_task.load('finance_corpus_summarized')
financebench_task_reranker = CrossEncoderReranker(queries = financebench_task.queries, corpus = financebench_task.corpus, reranker = model)
financebench_task_bm25_retriever = BM25_Retriever()
retrieved_result = financebench_task.retrieve(retriever = financebench_task_bm25_retriever, corpus = financebench_task.corpus, top_k = 50)
final_result = financebench_task_reranker.rerank(retrieved_result, top_k = 10)
financebench_task.save_retrieved_results(retrieved_result = final_result)


# ConvFinQA Task
print(f"ConvFinQA task")
convfinqa_task = ConvFinQATask()
convfinqa_task.load('finance_corpus_summarized')
convfinqa_task_reranker = CrossEncoderReranker(queries = convfinqa_task.queries, corpus = convfinqa_task.corpus, reranker = model)
convfinqa_task_bm25_retriever = BM25_Retriever()
retrieved_result = convfinqa_task.retrieve(retriever = convfinqa_task_bm25_retriever, corpus = convfinqa_task.corpus, top_k = 50)
final_result = convfinqa_task_reranker.rerank(retrieved_result, top_k = 10)
convfinqa_task.save_retrieved_results(retrieved_result = final_result)


# FinQABench Task
print(f"FinQABench task")
finqabench_task = FinQABenchTask()
finqabench_task.load('finance_corpus_summarized')
finqabench_task_reranker = CrossEncoderReranker(queries = finqabench_task.queries, corpus = finqabench_task.corpus, reranker = model)
finqabench_task_bm25_retriever = BM25_Retriever()
retrieved_result = finqabench_task.retrieve(retriever = finqabench_task_bm25_retriever, corpus = finqabench_task.corpus, top_k = 50)
final_result = finqabench_task_reranker.rerank(retrieved_result, top_k = 10)
finqabench_task.save_retrieved_results(retrieved_result = final_result)


# TATQA Task
print(f"TATQA task")
tatqa_task = TATQATask()
tatqa_task.load('finance_corpus_summarized')
tatqa_task_reranker = CrossEncoderReranker(queries = tatqa_task.queries, corpus = tatqa_task.corpus, reranker = model)
tatqa_task_bm25_retriever = BM25_Retriever()
retrieved_result = tatqa_task.retrieve(retriever = tatqa_task_bm25_retriever, corpus = tatqa_task.corpus, top_k = 50)
final_result = tatqa_task_reranker.rerank(retrieved_result, top_k = 10)
tatqa_task.save_retrieved_results(retrieved_result = final_result)


# FinDER Task
print(f"FinDER task")
finder_task = FinDERTask()
finder_task.load('finance_corpus_summarized')
finder_task_reranker = CrossEncoderReranker(queries = finder_task.queries, corpus = finder_task.corpus, reranker = model)
finder_task_bm25_retriever = BM25_Retriever()
retrieved_result = finder_task.retrieve(retriever = finder_task_bm25_retriever, corpus = finder_task.corpus, top_k = 50)
final_result = finder_task_reranker.rerank(retrieved_result, top_k = 10)
finder_task.save_retrieved_results(retrieved_result = final_result)

MultiHeirtt task
Saved result successfully to ./financerag_result/multiheirtt_result.csv!
FinQA task
Saved result successfully to ./financerag_result/finqa_result.csv!
FinanceBench task
Saved result successfully to ./financerag_result/financebench_result.csv!
ConvFinQA task
Saved result successfully to ./financerag_result/convfinqa_result.csv!
FinQABench task
Saved result successfully to ./financerag_result/finqabench_result.csv!
TATQA task
Saved result successfully to ./financerag_result/tatqa_result.csv!
FinDER task
Saved result successfully to ./financerag_result/finder_result.csv!


In [25]:
os.mkdir('financerag_result/helllo')

In [17]:
len(retrieved_result['q00001'])

50